# Human Annotation with the Galileo Python SDK

This notebook shows how to add **human feedback** to Galileo traces. Automatic metrics are useful for scale, but annotation is the right tool when you need people to judge qualities like empathy, helpfulness, policy fit, or whether a response is ready to ship.

In Galileo, annotations are reusable review forms attached to Sessions, Traces, or Spans. You first create annotation templates, then reviewers submit ratings against logged records. Galileo supports score, star, tags/categories, text, and thumbs up/down annotations. The Logs and Messages views can surface these annotations, and annotations can be exported as columns for downstream analysis.

Use this pattern when:

- QA reviewers need to label production traces
- domain experts need to score outputs that automatic metrics cannot judge reliably
- you want to turn reviewed logs into datasets for prompt iteration or model tuning


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


In [2]:
import requests

from galileo import GalileoLogger, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream
from galileo.projects import delete_project, get_project

PROJECT_NAME = 'GalileoEval_S7_Annotations'
LOG_STREAM_NAME = 'annotation-review'
MODEL = 'gpt-4o-mini'


## 1. Initialize Galileo

Initialize a project and log stream for the annotation demo, then print the console URLs so you can inspect the records after the notebook runs.


In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


https://console.demo-v2.galileocloud.io/project/716291e7-a593-488f-bd61-ed84452a669e
https://console.demo-v2.galileocloud.io/project/716291e7-a593-488f-bd61-ed84452a669e/log-streams/05eff68e-bf91-4e90-aeb0-b2431465fcb8


## 2. Log Traces That Need Human Review

Annotations work best once you already have records worth reviewing. This step logs a few customer-support style traces and captures their trace IDs so a reviewer can rate them later.


In [4]:
review_samples = [
    {
        'input': 'Draft a refund email for a frustrated customer.',
        'output': 'We are sorry for the inconvenience. We will issue a refund within 5 business days.',
    },
    {
        'input': 'Write a short delivery-delay update for a premium customer.',
        'output': 'Your order is delayed. We will send another update soon.',
    },
    {
        'input': 'Respond to a user asking if their account deletion request is complete.',
        'output': 'Your account deletion request has been received and will be completed within 72 hours.',
    },
]

logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
review_examples = []

for sample in review_samples:
    logger.start_trace(input=sample['input'])
    logger.add_llm_span(
        input=[Message(role=MessageRole.user, content=sample['input'])],
        output=Message(role=MessageRole.assistant, content=sample['output']),
        model=MODEL,
    )
    trace_id = str(logger.traces[-1].id)
    logger.conclude(output=sample['output'])
    review_examples.append({**sample, 'trace_id': trace_id})

logger.flush()
review_examples


[{'input': 'Draft a refund email for a frustrated customer.',
  'output': 'We are sorry for the inconvenience. We will issue a refund within 5 business days.',
  'trace_id': 'da10c0ea-45f1-4483-9f53-1681911516f6'},
 {'input': 'Write a short delivery-delay update for a premium customer.',
  'output': 'Your order is delayed. We will send another update soon.',
  'trace_id': '0b2005d5-1b55-437a-8fb8-c595e4356ef1'},
 {'input': 'Respond to a user asking if their account deletion request is complete.',
  'output': 'Your account deletion request has been received and will be completed within 72 hours.',
  'trace_id': 'a6e71cee-3e33-4c6a-be4a-911d7bb376de'}]

## 3. Create Annotation Templates

Annotation templates define the reusable review forms that appear in Galileo. This notebook creates five common manual-feedback types:

- `score` for a numeric helpfulness rating
- `star` for a 1-5 quality signal
- `tags` for category-style issue labels
- `text` for a reviewer note
- `like_dislike` for a fast ship / do-not-ship signal

The logging SDK handles trace creation, while the public annotation endpoints let you configure templates and submit ratings.


In [5]:
config = GalileoPythonConfig.get()
project = get_project(name=PROJECT_NAME)
project_id = str(project.id)
base_url = str(config.api_url).rstrip('/')
headers = {'Galileo-API-Key': config.api_key.get_secret_value()}

template_specs = [
    {
        'name': 'helpfulness_review',
        'criteria': 'Does the response help the customer clearly and politely?',
        'include_explanation': True,
        'constraints': {'annotation_type': 'score', 'min': 1, 'max': 5},
    },
    {
        'name': 'tone_star',
        'criteria': 'Rate the response tone from 1 to 5 stars.',
        'include_explanation': False,
        'constraints': {'annotation_type': 'star'},
    },
    {
        'name': 'issue_tags',
        'criteria': 'Tag quality issues you notice in the response.',
        'include_explanation': False,
        'constraints': {
            'annotation_type': 'tags',
            'tags': ['missing_empathy', 'too_vague', 'unsafe'],
            'allow_other': False,
        },
    },
    {
        'name': 'review_note',
        'criteria': 'Leave a reviewer note for follow-up.',
        'include_explanation': False,
        'constraints': {'annotation_type': 'text'},
    },
    {
        'name': 'ship_it',
        'criteria': 'Would you approve this response for production?',
        'include_explanation': True,
        'constraints': {'annotation_type': 'like_dislike'},
    },
]

existing_templates = requests.get(
    f'{base_url}/v2/projects/{project_id}/annotation/templates',
    headers=headers,
    timeout=30,
)
existing_templates.raise_for_status()
existing_by_name = {item['name']: item for item in existing_templates.json()}

annotation_templates = {}
for spec in template_specs:
    if spec['name'] in existing_by_name:
        annotation_templates[spec['name']] = existing_by_name[spec['name']]
        continue

    response = requests.post(
        f'{base_url}/v2/projects/{project_id}/annotation/templates',
        headers=headers,
        json=spec,
        timeout=30,
    )
    response.raise_for_status()
    annotation_templates[spec['name']] = response.json()

{
    name: {
        'id': template['id'],
        'annotation_type': template['constraints']['annotation_type'],
        'usage_count': template['usage_count'],
    }
    for name, template in annotation_templates.items()
}


{'helpfulness_review': {'id': '7077bb14-ac2a-48ae-a2e8-39b8b7e0dd33',
  'annotation_type': 'score',
  'usage_count': 3},
 'tone_star': {'id': 'a881a825-5f3d-48cb-81a1-334cfdace3cb',
  'annotation_type': 'star',
  'usage_count': 1},
 'issue_tags': {'id': 'd65d48f2-a4e5-4b14-a8bf-169d98c1f8c9',
  'annotation_type': 'tags',
  'usage_count': 1},
 'review_note': {'id': '1a063de2-eaf5-4e90-81a6-b36af1ed6101',
  'annotation_type': 'text',
  'usage_count': 1},
 'ship_it': {'id': 'f43f410e-7e66-4d32-9ff3-567ede5bbaf5',
  'annotation_type': 'like_dislike',
  'usage_count': 1}}

## 4. Submit Human Ratings to a Trace

A rating is attached to a specific record ID. In this notebook we annotate a **trace**, which is a good fit for one prompt/response pair. The same concept can also be applied to Sessions or Spans.


In [6]:
target_trace = review_examples[0]

rating_payloads = {
    'helpfulness_review': {
        'explanation': 'Polite and actionable, but it could acknowledge the customer frustration more explicitly.',
        'rating': {'annotation_type': 'score', 'value': 4},
    },
    'tone_star': {
        'rating': {'annotation_type': 'star', 'value': 4},
    },
    'issue_tags': {
        'rating': {'annotation_type': 'tags', 'value': ['missing_empathy']},
    },
    'review_note': {
        'rating': {'annotation_type': 'text', 'value': 'Add one sentence that validates the customer frustration before promising the refund.'},
    },
    'ship_it': {
        'explanation': 'Close to acceptable, but the tone needs one more revision before production.',
        'rating': {'annotation_type': 'like_dislike', 'value': False},
    },
}

annotation_results = {}
for template_name, body in rating_payloads.items():
    template_id = annotation_templates[template_name]['id']
    response = requests.put(
        f'{base_url}/v2/projects/{project_id}/annotation/templates/{template_id}/records/{target_trace["trace_id"]}/rating',
        headers=headers,
        json=body,
        timeout=30,
    )
    response.raise_for_status()
    annotation_results[template_name] = response.json()

{
    'trace_id': target_trace['trace_id'],
    'annotations': annotation_results,
}


{'trace_id': 'da10c0ea-45f1-4483-9f53-1681911516f6',
 'annotations': {'helpfulness_review': {'explanation': 'Polite and actionable, but it could acknowledge the customer frustration more explicitly.',
   'rating': {'annotation_type': 'score', 'value': 4},
   'created_at': '2026-03-14T03:21:37.000716Z',
   'created_by': '7680b8ce-69f8-40ea-8238-d7abcc80149f'},
  'tone_star': {'explanation': None,
   'rating': {'annotation_type': 'star', 'value': 4},
   'created_at': '2026-03-14T03:21:38.506334Z',
   'created_by': '7680b8ce-69f8-40ea-8238-d7abcc80149f'},
  'issue_tags': {'explanation': None,
   'rating': {'annotation_type': 'tags', 'value': ['missing_empathy']},
   'created_at': '2026-03-14T03:21:39.839477Z',
   'created_by': '7680b8ce-69f8-40ea-8238-d7abcc80149f'},
  'review_note': {'explanation': None,
   'rating': {'annotation_type': 'text',
    'value': 'Add one sentence that validates the customer frustration before promising the refund.'},
   'created_at': '2026-03-14T03:21:41.2471

## 5. Inspect the Templates and Next Steps

You can use the template list to confirm the annotation types now configured for the project and to inspect `usage_count` after ratings are submitted.

In the Galileo Console:

- the **Logs** page can show annotations as columns
- the **Messages** view highlights available annotations and lets reviewers submit them manually
- **Annotation Queues** are an Enterprise Beta option for routing records to human reviewers at scale


In [7]:
template_list = requests.get(
    f'{base_url}/v2/projects/{project_id}/annotation/templates',
    headers=headers,
    timeout=30,
)
template_list.raise_for_status()

[
    {
        'name': template['name'],
        'annotation_type': template['constraints']['annotation_type'],
        'usage_count': template['usage_count'],
    }
    for template in template_list.json()
]


[{'name': 'helpfulness_review', 'annotation_type': 'score', 'usage_count': 4},
 {'name': 'issue_tags', 'annotation_type': 'tags', 'usage_count': 2},
 {'name': 'review_note', 'annotation_type': 'text', 'usage_count': 2},
 {'name': 'ship_it', 'annotation_type': 'like_dislike', 'usage_count': 2},
 {'name': 'tone_star', 'annotation_type': 'star', 'usage_count': 2}]

## 6. Optional Cleanup

Keep the project around if you want to inspect the annotations in the Galileo Console. When you are done, run the cell below to delete the demo project.


In [ ]:
try:
    delete_project(name=PROJECT_NAME)
except Exception:
    pass

'Deleted annotation demo project'
